<a href="https://colab.research.google.com/github/Kamaumbugua-dev/Kamaumbugua-dev/blob/main/Route_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# -----------------------------
# DATA GENERATION
# -----------------------------
def generate_data(n=1200):
    data = []
    for _ in range(n):
        distance = np.random.uniform(2, 21)   # km
        hour = np.random.choice(range(6, 22))
        traffic = np.random.uniform(0.2, 1.0)
        rainy = np.random.choice([0, 1], p=[0.7, 0.3]) # 30% chance rain

        # ---- Fare Logic ----
        if distance < 7:
            fare = np.random.randint(50, 201)
        else:
            fare = min(350, 50 * int(distance))

        if rainy == 1:
            fare += np.random.randint(150, 401)
            fare = min(fare, 400)

        if (7 <= hour <= 9) or (17 <= hour <= 20): # peak hours
            fare *= 1.2

        # Round fare to nearest 10
        fare = int(round(fare / 10.0) * 10)

        # Cost model
        cost = distance * 10 + (5 if distance >= 15 else 0)
        profit = max(50, fare - cost)  # ensure minimum profit is 50

        data.append([distance, hour, traffic, rainy, fare, profit])

    return pd.DataFrame(data, columns=["distance", "hour", "traffic", "rainy", "fare", "profit"])

# -----------------------------
# TRAIN MODEL
# -----------------------------
df = generate_data(1200)

X = df[["distance", "hour", "traffic", "rainy", "fare"]]
y = df["profit"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
route_model = RandomForestRegressor(n_estimators=200, random_state=42)
route_model.fit(X_train, y_train)

y_pred = route_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"✅ Model trained | Average error (MAE): {mae:.0f} KSh")

# -----------------------------
# SIMULATION
# -----------------------------
def simulate_daily_profit(model, trips=12, strategy="mixed"):
    trip_results = []

    for i in range(trips):
        distance = np.random.uniform(2, 21)
        hour = np.random.choice(range(6, 22))
        traffic = np.random.uniform(0.2, 1.0)
        rainy = np.random.choice([0, 1], p=[0.7, 0.3])

        # Strategy filter
        if strategy == "short" and distance >= 7:
            continue
        if strategy == "long" and distance < 7:
            continue
        if strategy == "rainy" and rainy == 0:
            continue

        # ---- Fare Logic ----
        if distance < 7:
            fare = np.random.randint(50, 201)
        else:
            fare = min(350, 50 * int(distance))

        if rainy == 1:
            fare += np.random.randint(150, 401)
            fare = min(fare, 400)

        if (7 <= hour <= 9) or (17 <= hour <= 20):
            fare *= 1.2

        # Round to nearest 10
        fare = int(round(fare / 10.0) * 10)

        trip_data = pd.DataFrame([[distance, hour, traffic, rainy, fare]],
                                 columns=["distance", "hour", "traffic", "rainy", "fare"])
        profit = max(50, int(round(model.predict(trip_data)[0])))

        trip_results.append({"Trip": i+1,
                             "Distance(km)": round(distance,1),
                             "Hour": hour,
                             "Rainy": "Yes" if rainy else "No",
                             "Fare(KSh)": fare,
                             "Profit(KSh)": profit})

    df_trips = pd.DataFrame(trip_results)
    if df_trips.empty:
        return None, 0, None, None

    total_profit = df_trips["Profit(KSh)"].sum()
    best_trip = df_trips.loc[df_trips["Profit(KSh)"].idxmax()]
    worst_trip = df_trips.loc[df_trips["Profit(KSh)"].idxmin()]

    return df_trips, total_profit, best_trip, worst_trip

# -----------------------------
# MARKOV CHAIN ROUTE SUGGESTION
# -----------------------------
def suggest_next_route(current="short"):
    # Transition probabilities: [short, long, rainy]
    transitions = {
        "short":  {"short": 0.7, "long": 0.2, "rainy": 0.1},
        "long":   {"short": 0.2, "long": 0.6, "rainy": 0.2},
        "rainy":  {"short": 0.2, "long": 0.2, "rainy": 0.6},
    }
    probs = transitions[current]
    next_route = np.random.choice(list(probs.keys()), p=list(probs.values()))
    return next_route

# -----------------------------
# RIDER REPORT
# -----------------------------
def rider_report(strategy="mixed"):
    df_trips, total_profit, best_trip, worst_trip = simulate_daily_profit(route_model, trips=12, strategy=strategy)
    if df_trips is None:
        print("⚠️ No trips available under this strategy today.")
        return

    avg_profit = df_trips["Profit(KSh)"].mean()

    print("\n================= 📊 DAILY RIDER REPORT =================")
    print(df_trips.to_string(index=False))
    print("\n💰 Total Profit:", int(total_profit), "KSh")
    print("📈 Average Profit per Trip:", int(avg_profit), "KSh")
    print(f"🏆 Best Trip: {best_trip['Profit(KSh)']} KSh | {best_trip['Distance(km)']} km at {best_trip['Hour']}:00 | Rain: {best_trip['Rainy']}")
    print(f"⚠️ Worst Trip: {worst_trip['Profit(KSh)']} KSh | {worst_trip['Distance(km)']} km at {worst_trip['Hour']}:00 | Rain: {worst_trip['Rainy']}")

    # Markov suggestion
    current_route = strategy if strategy != "mixed" else "short"
    suggested_next = suggest_next_route(current=current_route)
    print(f"\n🔮 Suggested Next Route (Markov): {suggested_next.upper()} route")

    # Rider-friendly advice
    print("\n🚀 TIPS TO IMPROVE EARNINGS:")
    if avg_profit < 100:
        print("- Avoid very short trips outside peak hours, they pay less.")
    if best_trip["Hour"] in [7,8,17,18,19,20]:
        print("- Peak hours are the most profitable. Ride more during morning/evening rush.")
    if (df_trips["Rainy"] == "Yes").sum() > 0:
        print("- Rainy rides bring higher pay. If safe, accept them.")
    if (df_trips["Distance(km)"] >= 15).sum() > 0:
        print("- Longer routes give more fare, but watch fuel/maintenance costs.")
    print("==========================================================")

# -----------------------------
# RUN DIFFERENT STRATEGIES
# -----------------------------
for strat in ["mixed", "short", "long", "rainy"]:
    print(f"\n\n===== STRATEGY: {strat.upper()} =====")
    rider_report(strategy=strat)

✅ Model trained | Average error (MAE): 1 KSh


===== STRATEGY: MIXED =====

================= 📊 DAILY RIDER REPORT =================
 Trip  Distance(km)  Hour Rainy  Fare(KSh)  Profit(KSh)
    1          20.4     6    No        350          141
    2          11.6    18    No        420          304
    3          12.3     8    No        420          297
    4           3.6    20   Yes        480          444
    5           7.0    13    No        350          278
    6          14.7    18    No        420          273
    7           9.4    15    No        350          256
    8           7.2    11    No        350          277
    9          12.6    13    No        350          224
   10           9.3     9    No        420          326
   11          13.5     6   Yes        400          265
   12           7.1    13   Yes        400          328

💰 Total Profit: 3413 KSh
📈 Average Profit per Trip: 284 KSh
🏆 Best Trip: 444 KSh | 3.6 km at 20:00 | Rain: Yes
⚠️ Worst Trip: 141 KSh | 20